In [1]:
import json

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
login(user_secrets.get_secret("HUGGINGFACE_ACCESS_TOKEN"))

In [2]:
MODEL_NAME = "Qwen/Qwen3-4B-Thinking-2507"
BASE_DIR = "/kaggle/input/datasets/prithvikarthik/shopping-tools/"
DATA_PATH = BASE_DIR + "shopping_tool_balanced.csv"
BASELINE_PATH = BASE_DIR + "shopping_tools.json"
PERTURBATIONS_PATH = BASE_DIR + "shopping_tool_perturbations.json"

device = "cuda" if torch.cuda.is_available() else "cpu"

df_balanced = pd.read_csv(DATA_PATH)
with open(BASELINE_PATH) as f:
    baseline_tools = json.load(f)
with open(PERTURBATIONS_PATH) as f:
    perturbations = json.load(f)

perturbation_types = [
    # "description_mismatch",
    # "paraphrase",
    # "negation_injection",
    # "vague_description",
    # "description_overload",
    # "suffix_noise_injection",
    # "prefix_noise_injection",
    # "interleaved_noise_injection",
]

In [3]:
print("Loading model and tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()

Loading model and tokenizer...


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 2560)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=2560, out_features=4096, bias=False)
          (k_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=2560, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (up_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (down_proj): Linear(in_features=9728, out_features=2560, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((2560,), eps=1e-06)
        (post_attention_layer

In [4]:
def run_eval(tool_desc_dict):
    """
    Runs inference using generation and shows progress bar.
    Returns accuracy and mean confidence.
    """

    tool_list = list(tool_desc_dict.keys())
    tool_text = "\n".join([f"{t}: {tool_desc_dict[t]}" for t in tool_list])

    correct_count = 0
    confidences = []
    all_errors = []

    system_prompt = f"""
You are a tool selection agent.

Select the SINGLE best tool for the user query.

Available tools:
{tool_text}

Instructions:
- Return ONLY the tool name.
- Do NOT output explanations.

Valid answers:
{", ".join(tool_list)}
"""

    progress_bar = tqdm(
        df_balanced.iterrows(),
        total=len(df_balanced),
        desc="Running inference",
        ncols=100
    )

    for i, row in progress_bar:

        query = row["Query"]
        gold_tool = row["Tool"]

        prompt = f"""{system_prompt}

User query:
{query}

Answer:
"""

        inputs = tokenizer(prompt, return_tensors="pt").to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=4,
                do_sample=False,
                output_scores=True,
                return_dict_in_generate=True,
                pad_token_id=tokenizer.eos_token_id,
            )

        generated_tokens = outputs.sequences[0][inputs["input_ids"].shape[1]:]

        decoded = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
        decoded = decoded.split("\n")[0].strip()

        pred_tool = None
        for t in tool_list:
            if t.lower() in decoded.lower():
                pred_tool = t
                break

        token_probs = []
        for j, token_id in enumerate(generated_tokens):
            probs = torch.softmax(outputs.scores[j], dim=-1)
            prob = probs[0, token_id].item()
            token_probs.append(prob)

        confidence = np.mean(token_probs) if token_probs else 0
        confidences.append(confidence)

        if pred_tool == gold_tool:
            correct_count += 1
        else:
            all_errors.append({
                "query": query,
                "gold_tool": gold_tool,
                "predicted_tool": pred_tool,
                "model_output": decoded,
                "confidence": confidence
            })

        current_acc = correct_count / (i + 1)
        progress_bar.set_postfix({
            "acc": f"{current_acc:.3f}",
            "conf": f"{np.mean(confidences):.3f}"
        })

    if all_errors:
        pd.DataFrame(all_errors).to_csv("shopping_tool_errors.csv", index=False)

    accuracy = correct_count / len(df_balanced)
    mean_conf = np.mean(confidences)

    return accuracy, mean_conf

In [5]:
results = []

print("\nEvaluating Baseline...")
acc, conf = run_eval(baseline_tools)
results.append({"type": "baseline", "accuracy": acc, "confidence": conf})

# 2. Perturbations
# for p_type in perturbation_types:
#     if p_type not in perturbations:
#         continue

#     print(f"Evaluating Perturbation: {p_type}...")
#     type_accs, type_confs = [], []

#     # We aggregate across variants within the type
#     variants = perturbations[p_type][:10]
#     for variant in tqdm(variants):
#         acc, conf = run_eval(variant)
#         type_accs.append(acc)
#         type_confs.append(conf)

#     results.append(
#         {
#             "type": p_type,
#             "accuracy": np.mean(type_accs),
#             "confidence": np.mean(type_confs),
#         }
#     )


Evaluating Baseline...


Running inference: 100%|███████████████████| 188/188 [01:17<00:00,  2.43it/s, acc=0.617, conf=0.658]


In [6]:
summary_df = pd.DataFrame(results)
summary_df["acc_delta"] = summary_df["accuracy"] - summary_df.iloc[0]["accuracy"]
summary_df["conf_delta"] = summary_df["confidence"] - summary_df.iloc[0]["confidence"]

print("\n" + "=" * 30)
print("FINAL PERTURBATION STUDY")
print("=" * 30)
print(summary_df[["type", "accuracy", "confidence", "acc_delta", "conf_delta"]])

summary_df.to_csv("shopping_tool_perturbation_summary.csv", index=False)


FINAL PERTURBATION STUDY
       type  accuracy  confidence  acc_delta  conf_delta
0  baseline  0.617021     0.65802        0.0         0.0
